Importing required libraries 

In [ ]:
import torch 
from torch import nn 
import torchvision



Creating model

In [ ]:
class CNNSENetModel(nn.Module):
    """CNN with SE net"""
    def __init__(self):
        super().__init__()

        #Creating CNN
        self.block1 = nn.Sequential(
            nn.Conv2d(3,100,kernel_size=3, stride = 1, padding =1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size =3, stride =1)
                      
        )
        
        #adding SE net block
        self.SENet = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Linear(in_features = 100, out_features = 100//16),
            nn.ReLU(),
            #100 out features because it is for each 100 channels
            nn.Linear(in_features = 100//16, out_features =100),
            nn.Sigmoid()

        )

        #classifiying 
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Linear(in_features = 100, out_features =6)
        )

    #performing forward pass so the model is able to update its weights and biases
    def forward(self,x):
        x = self.block1(x)
        #getting features from convolution layer
        features = x
        #generating weights 
        attention = self.SENet(x)
        #reshaping, adding new channels to prevent shape mismatch
        attention = attention[:,:,None,None]
        #scale operation
        x = features * attention 
        x = self.classifier(x) 
        return x